In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from scipy.spatial import distance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Read the data from txt files to pandas DataFrame
# Data read as numpy.float64 scaled into the range [-1.0:1.0]
x_train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/UCIHAR_dataset/train/X_train.txt', sep='\s+', header=None)
y_train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/UCIHAR_dataset/train/y_train.txt', header=None)

x_test = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/UCIHAR_dataset/test/X_test.txt', sep='\s+', header=None)
y_test = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/UCIHAR_dataset/test/y_test.txt', header=None)

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_7495/1766460634.py:3: SyntaxWarning: invalid escape sequence '\s'
  x_train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/UCIHAR_dataset/train/X_train.txt', sep='\s+', header=None)
/tmp/ipykernel_7495/1766460634.py:6: SyntaxWarning: invalid escape sequence '\s'
  x_test = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/UCIHAR_dataset/test/X_test.txt', sep='\s+', header=None)


In [ ]:
# Convert pandas DataFrame PyTorch Tensor
train_samples_float = torch.tensor(x_train.values)
train_labels = torch.tensor(y_train.values).to(torch.uint8).flatten() - 1 # -1: Put the labels in range [0:5] instead the original [1:6]

test_samples_float = torch.tensor(x_test.values)
test_labels = torch.tensor(y_test.values).to(torch.uint8).flatten() - 1 # -1: Put the labels in range [0:5] instead the original [1:6]

In [ ]:
# Convert dataset to uint8
# Data read as numpy.float64 scaled into the range [-1.0:1.0]

scale_factor = 3
offset = scale_factor
train_samples = (train_samples_float * scale_factor).round().to(torch.uint8) + offset
test_samples = (test_samples_float * scale_factor).round().to(torch.uint8) + offset

# Create a TensorDataset and DataLoader
#dataloader = torch.utils.data.DataLoader(TensorDataset(train_samples), batch_size=2, shuffle=True)

In [ ]:
# Constants
DIMENSIONS = 8192
FEATURE_LEVELS = 201
FEATURES = 561
CLASSES = 6

In [ ]:
# Key-value encoding
def Encode(dimensions, sample, levels, indexes):
    encoded = torch.zeros(dimensions, dtype=torch.int32)

    for i in range(sample.shape[0]):
        encoded += levels[sample[i].item()] ^ indexes[i]

    # Addition with a majority vote threshold
    sample_size = sample.shape[0]
    encoded_bin = torch.where(encoded > (sample_size // 2), 1, 0)

    return encoded_bin.to(dtype=torch.int8), encoded

In [ ]:
# Seed generation

def BinaryTensor(n, size):

    bin = torch.zeros(size, dtype=torch.uint8)

    for i in range(size):
        bin[size - 1 - i] = n % 2
        n //= 2

    return bin

def ISOLET_HVBit(t):
    a = torch.zeros_like(t)

    a[0] = t[0] ^ t[1]
    a[1] = t[2] ^ t[3]
    a[2] = t[4] ^ t[5]
    a[3] = t[6] ^ t[7]
    a[4] = t[8] ^ t[9]
    a[5] = t[10] ^ t[11]
    a[6] = t[12] ^ t[13]

    #return a[0] ^ a[1] ^ a[2] ^ a[3] ^ a[4] ^ a[6]
    return t[0] ^ t[1] ^ t[2] ^ t[3] ^ t[4] ^ t[5] ^ t[6] ^ t[7] ^ t[8] ^ t[9] ^ t[12] ^ t[13]


gen_seed = torch.zeros(DIMENSIONS, dtype=torch.uint8)

size = 16

for i in range(DIMENSIONS):
    gen_seed[i] = ISOLET_HVBit(BinaryTensor(i, size).roll(-(i % 16)))


In [ ]:
# GENERATED SEED

# Create hypervectors table: each value has an hypervector
level_hvs_bin = torch.zeros([FEATURE_LEVELS, DIMENSIONS], dtype=torch.uint8)
seed = gen_seed

for level in range(FEATURE_LEVELS):
    level_hvs_bin[level] = seed.roll(level)

# Create hypervectors_table: each feature index has an hypervector
indexes_hvs_bin = torch.zeros([FEATURES, DIMENSIONS], dtype=torch.uint8)

# New seed
for i in range(DIMENSIONS):
   seed[i] = 1 if seed[i] == 0 else 0

for i in range(FEATURES):
    indexes_hvs_bin[i] = seed.roll(i)

In [ ]:
# Encode train set

TRAIN_SET_SIZE = 500
#TRAIN_SET_SIZE = 5000
#TRAIN_SET_SIZE = train_samples.shape[0] # Full train set
encoded_train_set = []
encoded_train_set.append(torch.zeros([TRAIN_SET_SIZE, DIMENSIONS], dtype=torch.int8))
encoded_train_set.append(torch.zeros([TRAIN_SET_SIZE, 1], dtype=torch.int8))

encoded_train_set[1] = train_labels

for i, sample in enumerate(train_samples[:TRAIN_SET_SIZE]):
    encoded_train_set[0][i], _ = Encode(DIMENSIONS, sample, level_hvs_bin, indexes_hvs_bin)

#torch.save(encoded_train_set[0], "train_D10000.pt")
#encoded_train_set[0] = torch.load('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/train_D10000.pt', weights_only = True)

train_set = {
    'samples' : encoded_train_set[0],
    'labels' : encoded_train_set[1]
}

In [ ]:
# Encode test set

TEST_SET_SIZE = 500
#TEST_SET_SIZE = 5000
#TEST_SET_SIZE = test_samples.shape[0] # Full train set
encoded_test_set = []
encoded_test_set.append(torch.zeros([TEST_SET_SIZE, DIMENSIONS], dtype=torch.int8))
encoded_test_set.append(torch.zeros([TEST_SET_SIZE, 1], dtype=torch.int8))

encoded_test_set[1] = test_labels

for i, sample in enumerate(test_samples[:TEST_SET_SIZE]):
    encoded_test_set[0][i], _ = Encode(DIMENSIONS, sample, level_hvs_bin, indexes_hvs_bin)

#torch.save(encoded_test_set[0], "test_D10000.pt")
#encoded_test_set[0] = torch.load('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/UCIHAR/test_D10000.pt', weights_only = True)

test_set = {
    'samples' : encoded_test_set[0],
    'labels' : encoded_test_set[1]

}

In [ ]:
# Create class hypervectors using different amounts of train samples

train_sizes = [TEST_SET_SIZE]

class_hvs = torch.zeros([len(train_sizes), CLASSES, DIMENSIONS], dtype=torch.int32)
class_hvs_bin = torch.zeros([len(train_sizes), CLASSES, DIMENSIONS], dtype=torch.int8)

for i, size in enumerate(train_sizes):

    encoded_samples = train_set['samples'][:size]
    labels = train_set['labels'][:size]

    # Create hypervector classes
    # Standard centroid-based classifier: this is the most basic version of the classifier, where it adds each sample’s hypervector to its corresponding class
    for sample, label in zip(encoded_samples, labels.tolist()):
        class_hvs[i][label] += sample

    # Number of added samples in each class
    class_count = torch.bincount(labels)

    # Binarization: majority vote threshold
    for cl in range(CLASSES):
        class_hvs_bin[i][cl] = torch.where(class_hvs[i][cl] > (class_count[cl] // 2), 1, 0)

    print(f'class_hvs[{i}]: {CLASSES} class hypervectors created from {train_sizes[i]} train samples')

class_hvs[0]: 6 class hypervectors created from 500 train samples


In [ ]:
def Tensor2Int(t):

    value = 0

    for i in range(len(t)):
        value += t[i].item() * 2 ** i

    return value

In [ ]:
delta_h = []

for j in range(DIMENSIONS):
    max = 0
    min = 1000000
    for k in class_hvs_bin[0]:
        if k[j] > max:
            max = k[j]

        if k[j] < min:
            min = k[j]

    delta_h.append(max - min)

delta_h = torch.tensor(delta_h)

In [ ]:
sample = train_samples[5]

hv_bin, hv = Encode(DIMENSIONS, sample, level_hvs_bin, indexes_hvs_bin)

top = 3200
top_idxs = delta_h.topk(top).indices

image_file = open(f'UCIHAR_sample.txt', "w+")
for level in sample.flatten():
    image_file.write(f'{level.item():02x}\n')
image_file.close()

indexes_file = open(f'UCIHAR_idxs.txt', "w+")
for idx in top_idxs:
    indexes_file.write(f'{idx:04x}\n')
indexes_file.close()

class_hvs_bin_t = class_hvs_bin[0].T

indexes_file = open(f'UCIHAR_hvs.txt', "w+")
for idx in top_idxs:
    indexes_file.write(f'{Tensor2Int(class_hvs_bin_t[idx]):02x}\n')
indexes_file.close()

In [ ]:
print(indexes_hvs_bin.shape)
print(indexes_hvs_bin[:20, 4856])

count = 0
for i in range(561):
    f = level_hvs_bin[sample[i].item()][4856].item()
    x = indexes_hvs_bin[i][4856]
    count +=  f ^ x
    print(f'i: {i}    f: {f} ^ x: {x}, count: {count}')


torch.Size([561, 8192])
tensor([0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       dtype=torch.uint8)
i: 0    f: 0 ^ x: 0, count: 0
i: 1    f: 0 ^ x: 1, count: 1
i: 2    f: 0 ^ x: 1, count: 2
i: 3    f: 1 ^ x: 1, count: 2
i: 4    f: 1 ^ x: 1, count: 2
i: 5    f: 1 ^ x: 0, count: 3
i: 6    f: 1 ^ x: 0, count: 4
i: 7    f: 1 ^ x: 0, count: 5
i: 8    f: 1 ^ x: 1, count: 5
i: 9    f: 1 ^ x: 0, count: 6
i: 10    f: 0 ^ x: 1, count: 7
i: 11    f: 0 ^ x: 1, count: 8
i: 12    f: 1 ^ x: 0, count: 9
i: 13    f: 1 ^ x: 0, count: 10
i: 14    f: 1 ^ x: 0, count: 11
i: 15    f: 1 ^ x: 0, count: 12
i: 16    f: 1 ^ x: 1, count: 12
i: 17    f: 1 ^ x: 0, count: 13
i: 18    f: 1 ^ x: 0, count: 14
i: 19    f: 1 ^ x: 0, count: 15
i: 20    f: 1 ^ x: 0, count: 16
i: 21    f: 1 ^ x: 1, count: 16
i: 22    f: 0 ^ x: 1, count: 17
i: 23    f: 0 ^ x: 0, count: 17
i: 24    f: 0 ^ x: 1, count: 18
i: 25    f: 0 ^ x: 1, count: 19
i: 26    f: 0 ^ x: 1, count: 20
i: 27    f: 0 ^ x: 0, count: 20
i: 28   

In [ ]:
similarity = torch.zeros([CLASSES])

for cl in range(CLASSES):
    similarity[cl] = 1 - distance.hamming(class_hvs_bin[0][cl][top_idxs], hv_bin[top_idxs])

print(similarity * len(top_idxs))


similarity = torch.zeros([CLASSES])

for cl in range(CLASSES):
    for i in top_idxs:
        similarity[cl] += 1 if class_hvs_bin[0][cl][i] == hv_bin[i] else 0

print(similarity, similarity.argmin())

tensor([1404., 1376., 1344., 2371., 2482., 2337.])
tensor([1404., 1376., 1344., 2371., 2482., 2337.]) tensor(2)
